# RUGD Traversability Classifier
Binary classification: **0 = non-traversable** | **1 = traversable**

**Expected Drive layout:**
```
MyDrive/RUGD_frames-with-annotations/
    RUGD_frames/
        creek/
            creek_00001.png
            ...
        park-2/
            park-2_00001.png
            ...
    RUGD_annotations/
        creek/
            creek_00001.png   ← color-coded label image
            ...
        RUGD_annotation-colormap.txt
```

**What you need:** Colab GPU runtime, no pip installs required.

In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


## 1 · Constants & Paths

In [2]:
import os

BASE        = "/content/drive/MyDrive/RUGD_frames-with-annotations"
FRAMES_DIR  = os.path.join(BASE, "RUGD_frames")
ANN_DIR     = os.path.join(BASE, "RUGD_annotations")
COLORMAP    = os.path.join(ANN_DIR, "RUGD_annotation-colormap.txt")
CACHE_FILE  = os.path.join(BASE, "labels_cache.pkl")

TARGET_SIZE = (96, 96)  # (W, H) for PIL.resize — model input is (96, 96, 1)

# Classes that make a region non-traversable.
# Names must match the second column in RUGD_annotation-colormap.txt exactly.
OBSTACLE_NAMES = {
    "tree", "pole", "water", "vehicle", "container/generic-object",
    "building", "rock-bed", "log", "bicycle", "person",
    "fence", "bush", "sign", "rock", "bridge", "picnic-table",
}

samplingRate = 3 # samples every 3 frames (per scene)

## 2 · Parse Colormap

Each annotation PNG pixel's RGB value maps to a semantic class.  
We build a fast lookup: a set of packed uint32 values for all obstacle colours.

In [3]:
import numpy as np

def load_colormap(path: str) -> dict:
    """Return {class_name: (R, G, B)} from the RUGD colormap txt file."""
    colormap = {}
    with open(path) as fh:
        for line in fh:
            parts = line.strip().split()
            if len(parts) < 5:
                continue
            # format: <id> <name> <R> <G> <B>
            name = parts[1]
            r, g, b = int(parts[2]), int(parts[3]), int(parts[4])
            colormap[name] = (r, g, b)
    return colormap


def pack_rgb(r: int, g: int, b: int) -> int:
    """Pack (R, G, B) into a single uint32 for fast set lookup."""
    return (r << 16) | (g << 8) | b


colormap = load_colormap(COLORMAP)

# Packed uint32 values for every obstacle colour
OBSTACLE_PACKED = set(
    pack_rgb(*colormap[name])
    for name in OBSTACLE_NAMES
    if name in colormap
)

missing = OBSTACLE_NAMES - colormap.keys()
if missing:
    print(f"WARNING: these obstacle names not found in colormap: {missing}")

print(f"Tracking {len(OBSTACLE_PACKED)} obstacle colours.")
for name in sorted(OBSTACLE_NAMES & colormap.keys()):
    r, g, b = colormap[name]
    print(f"  {name:30s} RGB({r:3d}, {g:3d}, {b:3d})")

Tracking 16 obstacle colours.
  bicycle                        RGB(  0, 255, 128)
  bridge                         RGB(102, 255, 255)
  building                       RGB(255,   0,   0)
  bush                           RGB(255, 153, 204)
  container/generic-object       RGB(255,   0, 127)
  fence                          RGB(102,   0, 204)
  log                            RGB(102,   0,   0)
  person                         RGB(204, 153, 255)
  picnic-table                   RGB(114,  85,  47)
  pole                           RGB(  0, 153, 153)
  rock                           RGB(153, 204, 255)
  rock-bed                       RGB(102, 102,   0)
  sign                           RGB(  0, 102, 102)
  tree                           RGB(  0, 255,   0)
  vehicle                        RGB(255, 255,   0)
  water                          RGB(  0, 128, 255)


## 3 · Build File Registry

Walk the frames directory and pair every image with its annotation.

In [4]:
from pathlib import Path

# Each entry: (scene_name, frame_path, annotation_path)
registry = []
missing_annotations = []

for scene_dir in sorted(Path(FRAMES_DIR).iterdir()):
    if not scene_dir.is_dir():
        continue
    scene = scene_dir.name
    for frame_path in sorted(scene_dir.glob("*.png")):
        ann_path = Path(ANN_DIR) / scene / frame_path.name
        if ann_path.exists():
            registry.append((scene, str(frame_path), str(ann_path)))
        else:
            missing_annotations.append(str(frame_path))

scenes = sorted(set(r[0] for r in registry))
print(f"{len(registry)} frame/annotation pairs across {len(scenes)} scenes.")
if missing_annotations:
    print(f"WARNING: {len(missing_annotations)} frames have no annotation — skipped.")
print(f"Scenes: {scenes}")

7436 frame/annotation pairs across 18 scenes.
Scenes: ['creek', 'park-1', 'park-2', 'park-8', 'trail', 'trail-10', 'trail-11', 'trail-12', 'trail-13', 'trail-14', 'trail-15', 'trail-3', 'trail-4', 'trail-5', 'trail-6', 'trail-7', 'trail-9', 'village']


## 4 · Label Cache

**Labelling rule:** if ≥1% of the bottom quarter of the frame is covered by obstacle pixels → **class 0** (non-traversable).  
Class-0 frames are augmented ×3 (hflip, vflip, rot90) in the label map to ease class imbalance.  

Annotation PNGs are colour-coded — each pixel's RGB value identifies its semantic class.

In [5]:
import pickle
from PIL import Image


def obstacle_mask_packed(ann_array: np.ndarray) -> np.ndarray:
    """
    Return a boolean (H, W) mask that is True wherever the annotation
    pixel belongs to an obstacle class.
    Packs each RGB triple into a single uint32 then does a vectorised isin check.
    """
    r = ann_array[..., 0].astype(np.uint32)
    g = ann_array[..., 1].astype(np.uint32)
    b = ann_array[..., 2].astype(np.uint32)
    packed = (r << 16) | (g << 8) | b
    return np.isin(packed, list(OBSTACLE_PACKED))


def compute_label(ann_path: str) -> int:
    """Return 0 (non-traversable) or 1 (traversable) from an annotation PNG."""
    ann = np.array(Image.open(ann_path).convert("RGB"))
    H, W = ann.shape[:2]

    path_start_vertical  = int(H * 0.67)          # top row of the bottom 50 %
    path_start_horizontal = int(W / 6)
    path_end_horizontal = int(W * (5/6))
    path_pixels = (H - path_start_vertical) * (path_end_horizontal - path_start_horizontal)  # total pixels in that region

    obstacle_pixels = obstacle_mask_packed(ann[path_start_vertical:, path_start_horizontal:path_end_horizontal]).sum()
    return 0 if (obstacle_pixels / path_pixels) >= 0.05 else 1

# label_map: (scene, frame_filename) → label
# aug_label_map: (scene, frame_filename, aug_suffix) → label  (class-0 only)
label_map = {}   # key: (scene, fname)         value: 0 or 1
frame_lookup = {}  # key: (scene, fname)        value: (frame_path, ann_path)

scene_counters = {} #used to sample every third per scene

for i, (scene, frame_path, ann_path) in enumerate(registry):
    if scene not in scene_counters:
      scene_counters[scene] = 0
    else:
      scene_counters[scene] += 1

    if(scene_counters[scene] % samplingRate != 0):
      continue


    fname = os.path.basename(frame_path)
    key   = (scene, fname)

    label = compute_label(ann_path)
    label_map[key]    = label
    frame_lookup[key] = (frame_path, ann_path)

    # Augment non-traversable frames to balance classes
    if label == 1:
        for aug in ("hflipAndZoomAndRotate"):
            akey = (scene, fname, aug)
            label_map[akey]    = 0
            frame_lookup[akey] = (frame_path, ann_path)

    if (i + 1) % 200 == 0:
        print(f"  {i+1}/{len(registry)}")

with open(CACHE_FILE, "wb") as fh:
    pickle.dump((label_map, frame_lookup), fh)
print("Cache saved.")


with open(CACHE_FILE, "rb") as fh:
    label_map, frame_lookup = pickle.load(fh)
print(f"Cache loaded — {len(label_map)} entries.")


count_0 = sum(1 for l in label_map.values() if l == 0)
count_1 = sum(1 for l in label_map.values() if l == 1)
print(f"Class 0 (non-traversable): {count_0}  ({100*count_0/len(label_map):.1f}%)")
print(f"Class 1 (traversable):     {count_1}  ({100*count_1/len(label_map):.1f}%)")

  400/7436
  1200/7436
  1800/7436
  2600/7436
  3200/7436
  3600/7436
  4400/7436
  4600/7436
  5200/7436
  5600/7436
  6400/7436
  6800/7436
  7200/7436
Cache saved.
Cache loaded — 32530 entries.
Class 0 (non-traversable): 30527  (93.8%)
Class 1 (traversable):     2003  (6.2%)


## 5 · Scene-Based Train / Val / Test Split

Splitting by *scene* prevents data leakage — consecutive frames from the same scene are almost identical.

In [6]:
import random

random.seed(42)
shuffled_scenes = scenes.copy()
random.shuffle(shuffled_scenes)
n = len(shuffled_scenes)

trainSplitEnd = int(n * .8)

train_scenes = set(shuffled_scenes[:trainSplitEnd])
test_scenes  = set(shuffled_scenes[trainSplitEnd:])

# Separate base keys (scene, fname) from augmented keys (scene, fname, aug)
def key_scene(key):
    return key[0]  # scene is always first element

all_keys    = list(label_map.keys())
train_keys  = [k for k in all_keys if key_scene(k) in train_scenes]
test_keys   = [k for k in all_keys if key_scene(k) in test_scenes]

for lst in (train_keys, test_keys):
    random.shuffle(lst)

for name, keys in (("Train", train_keys), ("Test", test_keys)):
    c0 = sum(1 for k in keys if label_map[k] == 0)
    c1 = sum(1 for k in keys if label_map[k] == 1)
    total = c0 + c1
    print(f"{name:5s} {total:5d} entries  —  class 0: {c0} ({100*c0/total:.0f}%)  class 1: {c1} ({100*c1/total:.0f}%)")

Train 28285 entries  —  class 0: 26532 (94%)  class 1: 1753 (6%)
Test   4245 entries  —  class 0: 3995 (94%)  class 1: 250 (6%)


## 6 · tf.data Pipeline

> **Normalisation note:** images are divided by 255 → float32 in [0, 1].  
> The model and the TFLite calibration both expect this range.  
> At inference time on-device you **must** divide by 255 before applying the INT8 quant formula.  
> Feeding raw 0-255 values to the INT8 model will silently produce wrong predictions
> (values above 127 can't even be represented in int8 and will overflow).

In [ ]:
import math
import tensorflow as tf
from google.colab import files

def processDataset(datasetX, datasetY, keys, frame_lookup):
  for i in range(len(keys)):
    if len(keys[i]) == 3:
        scene, fname, aug = keys[i]
        base_key = (scene, fname)
    else:
        base_key = keys[i]
        aug      = None

    frame_path, _ = frame_lookup[base_key]

    if not os.path.exists(frame_path):
        raise FileNotFoundError(f"Missing frame: {frame_path}")

    # Load as grayscale, resize, normalise → [0, 1]
    ogImage = Image.open(frame_path)
    if aug == "hflipAndZoomAndRotate":
      # Flip
      ogImage = ogImage.transpose(Image.FLIP_LEFT_RIGHT)

      # Rotate ±0.5°
      angle = np.random.uniform(-0.5, 0.5)
      ogImage = ogImage.rotate(angle, resample=Image.BILINEAR)

      # Zoom (crop center)
      zoom = np.random.uniform(0.9, 1.0)
      w, h = ogImage.size
      new_w, new_h = int(w * zoom), int(h * zoom)

      left = (w - new_w) // 2
      top  = (h - new_h) // 2

      ogImage = ogImage.crop((left, top, left + new_w, top + new_h))

    processImage = ogImage.convert("L").resize(TARGET_SIZE)

    processImage = np.array(processImage, dtype=np.float32)

    label = label_map[keys[i]] # (1,)

    datasetX[i] = processImage
    datasetY[i] = label

    if(i % 10 == 0):
      print(f"Stil making progress! {i} out of {len(keys)} done!")

  return (datasetX, datasetY)


train_keys = [k for k in train_keys if len(k) == 2 or k[2] == "hflipAndZoomAndRotate"]
test_keys  = [k for k in test_keys  if len(k) == 2]


train_ds_x = np.zeros((len(train_keys), TARGET_SIZE[0], TARGET_SIZE[1]), dtype = np.float32)
test_ds_x = np.zeros((len(test_keys), TARGET_SIZE[0], TARGET_SIZE[1]), dtype = np.float32)

train_ds_y = np.zeros((len(train_keys)), dtype = np.int8)
test_ds_y = np.zeros((len(test_keys)), dtype =  np.int8)

finalTrainDsX, finalTrainDsY = processDataset(train_ds_x, train_ds_y, train_keys, frame_lookup)
finalTestDsX, finalTestDsY = processDataset(test_ds_x, test_ds_y, test_keys, frame_lookup)

np.savez("trainDatasetRUGD.npz", x=finalTrainDsX, y=finalTrainDsY)
np.savez("testDatasetRUGD.npz", x=finalTestDsX, y=finalTestDsY)

files.download("trainDatasetRUGD.npz")
files.download("testDatasetRUGD.npz")

Stil making progress! 0 out of 1990 done!
Stil making progress! 10 out of 1990 done!
Stil making progress! 20 out of 1990 done!
Stil making progress! 30 out of 1990 done!
Stil making progress! 40 out of 1990 done!
Stil making progress! 50 out of 1990 done!
Stil making progress! 60 out of 1990 done!
Stil making progress! 70 out of 1990 done!
Stil making progress! 80 out of 1990 done!
Stil making progress! 90 out of 1990 done!
Stil making progress! 100 out of 1990 done!
